In [133]:
import pandas as pd
from pathlib import Path

In [134]:
project_root = Path.cwd()

In [135]:
def pq_to_df(asset_name: str) -> pd.DataFrame:
    # 1. Target the raw data subdirectory for the specific stock asset
    raw_dir = project_root / "data" / "raw" / asset_name

    # Check if the folder even exists before searching it
    if not raw_dir.exists():
        print(f"⚠️ Target folder missing: {raw_dir}")
        return pd.DataFrame()
    
    # 2. Search for all matching Parquet files using wildcard patterns
    # This matches anything that fits: asset_name_raw_*.pq
    parquet_files = sorted(raw_dir.glob(f"{asset_name}_raw_*.pq"))
    # If no files are found, exit early with an empty DataFrame
    if not parquet_files:
        print(f"❌ No raw Parquet files located for asset: {asset_name}")
        return pd.DataFrame()
    
    # 3. Pull the absolute newest file (last element in the sorted list)
    latest_file_path = parquet_files[-1]
    print(f"📖 Reading newest raw file for {asset_name}: {latest_file_path.name}")
    try:
        # 4. Ingest into a Pandas DataFrame
        # pd.read_parquet accepts pathlib Path objects natively as strings
        df = pd.read_parquet(str(latest_file_path))
        return df
    except Exception as e:
        print(f"💥 Critical error reading Parquet file for {asset_name}: {e}")
        return pd.DataFrame()

In [145]:
df = pq_to_df("XAG")

📖 Reading newest raw file for XAG: XAG_raw_20261027_15.pq


In [150]:
def flatten_alphavantage_ohlcv(asset_name: str, df_raw: pd.DataFrame, asset_config: dict) -> pd.DataFrame:
    """
    Extracts nested time-series dictionary data from Alpha Vantage raw dataframes,
    flattens the structural schema, and casts numerical columns.
    """
    if df_raw.empty:
        return pd.DataFrame() 
    try:
        # 1. Look up the schema strategy from the asset configuration registry
        schema_type = asset_config.get("schema_type")
        # =====================================================================
        # STRATEGY A: COMMODITIES (XAG, XAU)
        # =====================================================================
        if schema_type == "commodity_gold":
            # Commodities from AlphaVantage come out of the API as a flat list under "data"
            # If your bronze layer loaded it straight into columns, it contains 'date' and 'value'
            if "data" in df_raw.columns:
                # If nested under a 'data' string column, unpack the dictionaries
                clean_series = df_raw["data"].dropna()
                df_cleaned = pd.DataFrame(list(clean_series))
            else:
                # If already flat tables with columns on disk, just copy it
                df_cleaned = df_raw.copy()
            
            # Standardize names: commodities call it 'value', but your metrics need 'close'
            if "value" in df_cleaned.columns:
                df_cleaned = df_cleaned.rename(columns={"value": "close"})

        # =====================================================================
        # STRATEGY B: EQUITIES (NVDA) - Your Original Logic preserved!
        # =====================================================================
        elif schema_type == "equity_daily":
            if "Time Series (Daily)" not in df_raw.columns:
                raise KeyError(f"Expected column 'Time Series (Daily)' was missing for equity {asset_name}")
                
            clean_series = df_raw["Time Series (Daily)"].dropna()
            df_cleaned = pd.DataFrame.from_dict(clean_series.to_dict(), orient="index")
            df_cleaned.columns = [col.split(". ")[-1].strip().replace(" ", "_") for col in df_cleaned.columns]
            df_cleaned.index.name = "date"
            df_cleaned = df_cleaned.reset_index()

        else:
            print(f"⚠️ Skipping {asset_name}: Unknown schema_type configuration structural strategy '{schema_type}'.")
            return pd.DataFrame()
        
        # =====================================================================
        # UNIFIED DATA ENRICHMENT (Runs for BOTH Equities and Commodities)
        # =====================================================================
        # Ensure lineage metadata is tracked cleanly at the front
        if "asset_name" not in df_cleaned.columns:
            df_cleaned.insert(0, "asset_name", asset_name)
            
        # Convert string numbers from the API into proper numeric metrics types
        numeric_cols = ["open", "high", "low", "close", "volume"]
        for col in numeric_cols:
            if col in df_cleaned.columns:
                df_cleaned[col] = pd.to_numeric(df_cleaned[col])
                
        print(f"✨ Transformed {len(df_cleaned)} rows smoothly for {asset_name} using '{schema_type}' engine.")
        return df_cleaned

    except Exception as e:
        print(f"❌ Failed parsing layout structure for {asset_name}: {e}")
        return pd.DataFrame()

In [ ]:
with open(file_path, "r") as file:
    asset_configjson.load(file)

❌ Failed parsing layout structure for XAG: 'Time Series (Daily)'


In [ ]:
import pandas as pd



In [ ]:
import pandas as pd



📖 Reading newest raw file for NVDA: NVDA_raw_20261026_14.pq
✨ Transformed 100 rows smoothly for NVDA!


C:\Users\Kah Yuen\AppData\Local\Temp\ipykernel_16624\285547854.py:20: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_enriched.insert(4, "day_of_week", df_enriched[date_column].dt.dayofweek)  # 0=Monday, 6=Sunday


In [144]:
df_clean

,asset_name,year,month,day,day_of_week,date,open,high,low,close,volume,pct_change
0,NVDA,2026,6,3,2,2026-06-03,221.720,222.82,214.51,214.75,160907001,NaN
1,NVDA,2026,6,4,3,2026-06-04,213.905,221.60,210.97,218.66,169022152,0.018207
2,NVDA,2026,6,5,4,2026-06-05,214.530,214.87,204.33,205.10,219655531,-0.062014
3,NVDA,2026,6,8,0,2026-06-08,210.180,210.47,206.00,208.64,138372837,0.017260
4,NVDA,2026,6,9,1,2026-06-09,210.615,211.40,199.34,208.19,180962450,-0.002157
...,...,...,...,...,...,...,...,...,...,...,...,...
95,NVDA,2026,10,19,0,2026-10-19,218.095,219.62,214.70,217.09,98684721,0.013823
96,NVDA,2026,10,20,1,2026-10-20,215.980,218.46,212.77,216.47,79609078,-0.002856
97,NVDA,2026,10,21,2,2026-10-21,220.960,222.19,217.66,221.97,87085733,0.025408
98,NVDA,2026,10,22,3,2026-10-22,217.130,218.88,211.26,212.21,125407615,-0.043970


,asset_name,date,open,high,low,close,volume
0,NVDA,2026-10-23,210.990,213.85,210.18,210.96,80189765
1,NVDA,2026-10-22,217.130,218.88,211.26,212.21,125407615
2,NVDA,2026-10-21,220.960,222.19,217.66,221.97,87085733
3,NVDA,2026-10-20,215.980,218.46,212.77,216.47,79609078
4,NVDA,2026-10-19,218.095,219.62,214.70,217.09,98684721
...,...,...,...,...,...,...,...
95,NVDA,2026-06-09,210.615,211.40,199.34,208.19,180962450
96,NVDA,2026-06-08,210.180,210.47,206.00,208.64,138372837
97,NVDA,2026-06-05,214.530,214.87,204.33,205.10,219655531
98,NVDA,2026-06-04,213.905,221.60,210.97,218.66,169022152


In [77]:
import duckdb
con = duckdb.connect("stock_trackers.db")

In [78]:
create_table_query = """
    CREATE TABLE IF NOT EXISTS stock_history (
        Date DATE,
        ticker VARCHAR,
        open DOUBLE,
        high DOUBLE,
        low DOUBLE,
        close DOUBLE,
        volume BIGINT,
        PRIMARY KEY (Date, ticker)
    );
"""
con.execute(create_table_query)

In [79]:
staging_table_name = f"staging_{symbol}"
con.execute(f"""
    CREATE TEMP TABLE {staging_table_name} 
    AS SELECT * FROM df_ohlcv_values_melted
""")

In [81]:
upsert_query = f"""
    INSERT OR REPLACE INTO stock_history 
    SELECT 
        Date::DATE, -- Explicit cast to ensure SQL date mapping
        ticker,
        open,
        high,
        low,
        close,
        volume
    FROM staging_{symbol};
"""
con.execute(upsert_query)

In [82]:
print("✅ Data pipeline complete: Silver Master table updated successfully!")

✅ Data pipeline complete: Silver Master table updated successfully!


In [90]:
print("\n--- Current Data Warehouse Preview ---")
preview_df = con.execute("""
    SELECT ticker, COUNT(*) as total_records, MIN(Date) as earliest_date, MAX(Date) as latest_date 
    FROM stock_history 
    GROUP BY ticker
""").df()
print(preview_df)


--- Current Data Warehouse Preview ---
  ticker  total_records earliest_date latest_date
0   NVDA            100    2026-06-03  2026-10-23


In [89]:
del_bad_trackers = con.execute("""
DELETE FROM stock_history
WHERE ticker IS NULL 
   OR ticker = 'UNKNOWN' 
   """)


In [58]:
# ============================================================================
# DATA QUALITY ASSURANCE LAYER (THE DE TESTING SUITE)
# ============================================================================
print("\n🔍 Running Data Warehouse Validation Tests...")

# Test 1: Ensure NO Null values sneaked into critical columns
null_check = con.execute("""
    SELECT COUNT(*) FROM stock_history 
    WHERE ticker IS NULL OR Date IS NULL OR close IS NULL
""").fetchone()[0]

if null_check == 0:
    print("  ✅ Pass: No NULL values detected in primary target keys.")
else:
    print(f"  ❌ Fail: Found {null_check} rows containing illegal NULL values!")


🔍 Running Data Warehouse Validation Tests...
  ✅ Pass: No NULL values detected in primary target keys.


In [60]:
# Test 2: Financial Sanity Check (Prices must be greater than zero)
price_check = con.execute("""
    SELECT COUNT(*) FROM stock_history 
    WHERE open <= 0 OR close <= 0
""").fetchone()[0]

if price_check == 0:
    print("  ✅ Pass: All trading prices are valid positive numbers.")
else:
    print(f"  ❌ Fail: Detected {price_check} rows with impossible negative/zero stock prices!")

  ✅ Pass: All trading prices are valid positive numbers.


In [62]:
# Test 3: Business Logic Check (High must always be >= Low)
bounds_check = con.execute("""
    SELECT COUNT(*) FROM stock_history 
    WHERE high < low
""").fetchone()[0]

if bounds_check == 0:
    print("  ✅ Pass: Trading boundaries align (High is always greater than or equal to Low).")
else:
    print(f"  ❌ Fail: Found {bounds_check} corrupted rows where High price was lower than Low price!")


  ✅ Pass: Trading boundaries align (High is always greater than or equal to Low).


In [92]:
# Test 4: Row Counts Match Pandas DataFrame
db_row_count = con.execute("SELECT COUNT(*) FROM stock_history").fetchone()[0]
pandas_row_count = len(df_ohlcv_values_melted)

print(f"\n📊 Summary Alignment Check:")
print(f"  -> Records sitting inside DuckDB Table: {db_row_count}")
print(f"  -> Records processed by your Python loop: {pandas_row_count}")

ConnectionException: Connection Error: Connection already closed!

In [91]:
con.close()